# Cross Validation

Cross-validation adalah teknik evaluasi model yang membagi data menjadi beberapa *fold* (bagian), lalu melatih dan menguji model secara bergantian di setiap fold. Hasilnya lebih **robust** dibanding train/test split biasa karena setiap data pernah menjadi data validasi.

## 1. Import Library

Library yang digunakan:
- **pandas** — manipulasi data tabular
- **sklearn.model_selection** — `cross_val_score` untuk evaluasi model dengan CV
- **sklearn.ensemble** — `RandomForestRegressor` sebagai model prediksi
- **sklearn.pipeline** — membungkus preprocessing + model dalam satu alur
- **sklearn.impute** — `SimpleImputer` untuk mengisi nilai yang hilang (*missing values*)
- **matplotlib** — visualisasi hasil eksperimen

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt 
%matplotlib inline


## 2. Load Data

Dataset yang digunakan adalah **Ames Housing Dataset** — dataset harga rumah. Kolom `Id` dijadikan sebagai index baris.

In [2]:
#Read the data
train_data = pd.read_csv("D:/file baru/mendata/Intermediate Machine Learning/train.csv", index_col='Id')
test_data = pd.read_csv("D:/file baru/mendata/Intermediate Machine Learning/test.csv", index_col='Id')

## 3. Persiapan Fitur (X) dan Target (y)

Langkah-langkah persiapan data:
1. Baris dengan nilai `SalePrice` kosong dihapus — target tidak boleh `null`
2. `y` = kolom target **SalePrice** (harga rumah)
3. `X` = hanya kolom **numerik** dari data training (total 36 kolom)
4. `X_test` = kolom numerik yang sama dari data test

In [3]:
# train split the data and make X variabel for training data and y variabel for Saleprice 
train_data.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = train_data.SalePrice
train_data.drop(['SalePrice'], axis=1, inplace=True)


numeric_cols = [col for col in train_data.columns if train_data[col].dtype in['int64', 'float64']]
X = train_data[numeric_cols].copy()
X_test = test_data[numeric_cols].copy()



Tampilkan 6 baris pertama dari fitur `X` untuk melihat struktur data:

In [4]:
X.head(6)

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,GarageArea,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold
Id,,,,,,,,,,,,,,,,,,,,,
1,60,65.0,8450,7,5,2003,2003,196.0,706,0,...,548,0,61,0,0,0,0,0,2,2008
2,20,80.0,9600,6,8,1976,1976,0.0,978,0,...,460,298,0,0,0,0,0,0,5,2007
3,60,68.0,11250,7,5,2001,2002,162.0,486,0,...,608,0,42,0,0,0,0,0,9,2008
4,70,60.0,9550,7,5,1915,1970,0.0,216,0,...,642,0,35,272,0,0,0,0,2,2006
5,60,84.0,14260,8,5,2000,2000,350.0,655,0,...,836,192,84,0,0,0,0,0,12,2008
6,50,85.0,14115,5,5,1993,1995,0.0,732,0,...,480,40,30,0,320,0,0,700,10,2009


## 4. Membangun Pipeline

**Pipeline** menggabungkan dua langkah dalam satu objek:

| Step | Komponen | Fungsi |
|------|----------|--------|
| 1 | `SimpleImputer()` | Mengisi *missing values* dengan nilai **mean** (default) |
| 2 | `RandomForestRegressor` | Model Random Forest dengan **50 pohon** |

> **Keuntungan Pipeline:** Mencegah *data leakage* karena imputasi dilakukan secara terpisah pada setiap fold saat cross-validation.

In [5]:
my_pipeline = Pipeline(steps=[
    ('preprocessor', SimpleImputer()),
    ('model', RandomForestRegressor(n_estimators=50, random_state=0))
])

## 5. Cross-Validation Dasar (5-Fold)

Data training dibagi menjadi **5 fold**. Model dilatih dan dievaluasi sebanyak 5 kali secara bergantian:

```
Iterasi 1: [F1=Validation] + [F2+F3+F4+F5=Training] → MAE₁
Iterasi 2: [F2=Validation] + [F1+F3+F4+F5=Training] → MAE₂
Iterasi 3: [F3=Validation] + [F1+F2+F4+F5=Training] → MAE₃
Iterasi 4: [F4=Validation] + [F1+F2+F3+F5=Training] → MAE₄
Iterasi 5: [F5=Validation] + [F1+F2+F3+F4=Training] → MAE₅

Hasil Akhir = rata-rata(MAE₁ ... MAE₅)
```

- `cv=5` → 5-fold cross-validation
- `scoring='neg_mean_absolute_error'` → sklearn mengembalikan nilai **negatif** MAE, dikali `-1` agar positif
- **Hasil: MAE ≈ 18.276** → rata-rata error prediksi harga rumah sekitar $18.276

In [6]:
# Multiply by -1 since sklearn calculates *negative* MAE
scores = -1 * cross_val_score(my_pipeline, X, y,
                              cv=5,
                              scoring='neg_mean_absolute_error')

print("Average MAE score:", scores.mean())

Average MAE score: 18276.410356164386


## 6. Fungsi `get_score()` — Mencari n_estimators Optimal

Fungsi ini menerima parameter `n_estimators` (jumlah pohon dalam Random Forest) dan mengembalikan **rata-rata MAE** menggunakan **3-fold CV**.

Tujuannya adalah menemukan nilai `n_estimators` yang paling efisien — cukup besar untuk akurasi baik, tapi tidak berlebihan sehingga komputasi terlalu mahal.

In [7]:
def get_score(n_estimators):
    my_model1 = Pipeline(steps=[
        ('preprocessor', SimpleImputer()),
        ('model', RandomForestRegressor(n_estimators=n_estimators, random_state=0))
    ])
    scores = -1 * cross_val_score(my_model1, X, y,
                                  cv=3,
                                  scoring='neg_mean_absolute_error')

    return scores.mean()

## 7. Eksperimen — Membandingkan Nilai n_estimators

Loop menguji 8 nilai `n_estimators`: **50, 100, 150, 200, 250, 300, 350, 400**.

Setiap nilai dievaluasi dengan 3-fold CV. Hasilnya (rata-rata MAE) disimpan dalam dictionary `results`.

In [8]:
results = {}
for i in range(1,9):
    results[50 * i] = get_score(50 * i)

## 8. Visualisasi Hasil

Grafik menampilkan hubungan antara **jumlah pohon (n_estimators)** vs **MAE**:
- Sumbu X → nilai `n_estimators` (50 hingga 400)
- Sumbu Y → rata-rata MAE dari 3-fold CV

Dari grafik ini dapat diamati titik di mana penambahan pohon tidak lagi memberikan penurunan error yang signifikan, sehingga kita dapat memilih nilai `n_estimators` yang **paling efisien**.

In [9]:
plt.plot(list(results.keys()), list(results.values()))
plt.show()